# 01 — Text Extraction
This notebook demonstrates how ClauseGuard extracts raw text from PDF, DOCX, and TXT contract files.

**Libraries used:** `pdfplumber`, `python-docx`

**Production file:** `backend/extractor.py`

In [ ]:
import sys
sys.path.append('..')

import pdfplumber
from docx import Document
import os

## 1. Extract text from a PDF

In [ ]:
def extract_from_pdf(file_path):
    """Extract text from a PDF file using pdfplumber."""
    text = ""
    with pdfplumber.open(file_path) as pdf:
        print(f"Total pages: {len(pdf.pages)}")
        for i, page in enumerate(pdf.pages):
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
                print(f"  Page {i+1}: {len(page_text)} characters extracted")
    return text.strip()

pdf_path = "../tests/sample_contract.pdf"
raw_text = extract_from_pdf(pdf_path)
print(f"\nTotal characters extracted: {len(raw_text)}")

In [ ]:
# Preview the first 600 characters of the extracted text
print("--- RAW EXTRACTED TEXT (first 600 chars) ---")
print(raw_text[:600])

## 2. Extract text from a DOCX file

In [ ]:
def extract_from_docx(file_path):
    """Extract text from a DOCX file using python-docx."""
    doc = Document(file_path)
    paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
    print(f"Total non-empty paragraphs found: {len(paragraphs)}")
    return "\n".join(paragraphs)

# Uncomment and set path to test a DOCX file:
# docx_text = extract_from_docx("../tests/sample_contract.docx")
# print(docx_text[:600])
print("(DOCX extraction ready — provide a .docx file to test)")

## 3. Extract text from a TXT file

In [ ]:
def extract_from_txt(file_path):
    """Read a plain text file."""
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

# Demo: write a temp file and read it back
temp_path = "/tmp/sample.txt"
with open(temp_path, "w") as f:
    f.write("This Agreement is made between Party A and Party B.\nAll IP shall transfer to the Client.")

txt_text = extract_from_txt(temp_path)
print(txt_text)

## 4. Unified `extract_text()` function

In [ ]:
def extract_text(file_path):
    """Dispatch to the right extractor based on file extension."""
    extension = os.path.splitext(file_path)[1].lower()
    if extension == ".pdf":
        return extract_from_pdf(file_path)
    elif extension == ".docx":
        return extract_from_docx(file_path)
    elif extension == ".txt":
        return extract_from_txt(file_path)
    else:
        raise ValueError(f"Unsupported file type: {extension}")

text = extract_text("../tests/sample_contract.pdf")
print(f"Extracted {len(text)} characters from PDF")
print(text[:400])

## 5. Observations & Issues Encountered

- **Line breaks in PDFs:** pdfplumber often breaks long sentences across lines because PDF layout is column-based. This requires the post-processing in `cleaner.py`.
- **Markdown artifacts:** Contracts saved as PDF from markdown editors retain `##` headers and `**bold**` markers — useful for clause detection.
- **No OCR:** pdfplumber reads text-layer PDFs only. Scanned/image PDFs need OCR (e.g. `pytesseract`) which is out of scope.